In [1]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

from battle_cat_calc.models import GachaSlot, Position, Track, GachaPool, GachaStep
from battle_cat_calc.simulator import GachaSimulator
from battle_cat_calc.fetcher import BCDataFetcher
from battle_cat_calc.parser import BCParser

from tqdm.notebook import tqdm

In [2]:
# Params

max_roll = 500

In [3]:
# PoolA

# 抓取資料
fetcher_a = BCDataFetcher()
html_a = await fetcher_a.fetch_html(seed="1234567890", event="2025-12-12_1019", lang="tw", count=max_roll)

# 解析
parser_a = BCParser(html_a)
track_a = parser_a.track_a
track_b = parser_a.track_b

# 建立池
pool_a = GachaPool(
    pool_id="2025-12-12_1019",
    name="2025-12-12 ~ 2025-12-22: 全新傳說稀有和超激稀有角色參戰！新世紀福音戰士合作轉蛋登場！",
    track_a=track_a,
    track_b=track_b,
    has_guaranteed=True
)

In [4]:
# Pool B

# 抓取資料
fetcher_b = BCDataFetcher()
html_b = await fetcher_b.fetch_html(seed="1234567890", event="2025-12-12_1020", lang="tw", count=max_roll)

# 解析
parser_b = BCParser(html_b)
track_a = parser_b.track_a
track_b = parser_b.track_b

# 建立池
pool_b = GachaPool(
    pool_id="2025-12-12_1020",
    name="2025-12-12 ~ 2025-12-22: 全新傳說稀有角色參戰！新世紀福音戰士合作轉蛋2nd登場！",
    track_a=track_a,
    track_b=track_b,
    has_guaranteed=True
)

In [5]:
# 模擬
sim = GachaSimulator([
    pool_a, 
    pool_b
])

targets = set([
    '月影少年薰',
    'EVA量產機',
    'EVA零號機',
    'EVA初號機',
    '真嗣貓',
    '白夜姬零',
    '第6使徒',
    '第10使徒',
    'EVA2號機',
    '通訊聯絡機零月',
    'EVA8號機',
    '空中戰艦貓咪Wunder',   
    '第4使徒',
    '第9使徒',
    '獨眼少女明日香',
    'EVA第13號機',
    # '小小零',
    # '小小明日香',
    # '小小美里',
    # '小小真理',
    # '小小律子',
    # '小小綾波零（暫稱）',
    # '零貓',
    # '明日香貓',
    # '真理貓',
    # '薰貓'
])

pbar = tqdm(total=max_roll, desc="Find Best Path", unit="rolls")
last_cost = 0

def update_progress(cost, processed, queue):
    global last_cost
    if cost > last_cost:
        pbar.update(cost - last_cost)
        pbar.set_postfix({"狀態數": processed, "佇列": queue})
        last_cost = cost

result = sim.find_best_path(
    Position(1, 'A'), 
    targets, 
    max_rolls=max_roll,
    progress_callback=update_progress
)
pbar.close()
# 顯示結果
print(result.to_summary())

Find Best Path:   0%|          | 0/500 [00:00<?, ?rolls/s]

=== 模擬結果 ===
總花費: 134 抽
路徑步數: 26 (單抽 14, 十連 12)
收集到: EVA2號機, EVA8號機, EVA初號機, EVA第13號機, EVA量產機, EVA零號機, 月影少年薰, 獨眼少女明日香, 白夜姬零, 真嗣貓, 空中戰艦貓咪Wunder, 第10使徒, 第4使徒, 第6使徒, 第9使徒, 通訊聯絡機零月
使用池子: 2025-12-12_1019, 2025-12-12_1020
起點: Position(id=1, track='A') → 終點: Position(id=144, track='A')

=== 路徑詳情 ===
1. [2025-12-12_1019] 1A → 12A: 白夜姬零 (十連) (跳軌)
2. [2025-12-12_1020] 12A → 22B: 第9使徒 (十連) (跳軌)
3. [2025-12-12_1019] 22B → 23B: 鬥牛士貓 (單抽)
4. [2025-12-12_1020] 23B → 24B: 真理貓 (單抽)
5. [2025-12-12_1019] 24B → 35A: EVA量產機 (十連) (跳軌)
6. [2025-12-12_1020] 35A → 36A: EVA第13號機 (單抽)
7. [2025-12-12_1020] 36A → 37A: 貓咪番長 (單抽)
8. [2025-12-12_1019] 37A → 38A: 弓箭手貓 (單抽)
9. [2025-12-12_1019] 38A → 49B: EVA零號機 (十連) (跳軌)
10. [2025-12-12_1020] 49B → 60B: 第4使徒 (十連) (跳軌)
11. [2025-12-12_1019] 60B → 71A: 真嗣貓 (十連) (跳軌)
12. [2025-12-12_1019] 71A → 72A: 第6使徒 (單抽)
13. [2025-12-12_1019] 72A → 82B: 第10使徒 (十連) (跳軌)
14. [2025-12-12_1020] 82B → 93A: EVA8號機 (十連) (跳軌)
15. [2025-12-12_1020] 93A → 103B: 通訊聯絡機零月 (十連) (跳軌)
16. [2025-12-

In [6]:
result

GachaResult(cost=134, steps=26, collected=16)